# 04 — Modeling: Baseline and Shortlist

Place the prepared **`train.csv`** in the same folder as this notebook, then run the cells from top to bottom. The first code cell reads `train.csv` directly. This notebook does not repeat the original data splitting or sampling and does not read `test.csv`.

Imputation, encoding, and scaling are fitted within each CV fold. The 2016 test sample remains reserved for final evaluation.

**Dependencies:** pandas, NumPy, scikit-learn >= 1.2, IPython, and threadpoolctl. Saved modeling results use the supplied preparation notebook's 150,000-loan training sample.

## 1. Modeling strategy

`Default = 1` represents a charged-off loan; `Default = 0` represents a fully paid loan. The FICO-only logistic regression establishes a simple reference. The five additional families are logistic regression, decision tree, random forest, Extra Trees, and histogram gradient boosting.

All models use identical stratified five-fold cross-validation splits within the 2007–2015 training sample. Imputation, encoding, and scaling are learned separately inside each fold. ROC-AUC is the primary ranking metric; average precision (AP), precision, recall, F1, accuracy, and Brier score are reported as mean ± sample standard deviation. Lower Brier scores are better; higher values are better for the other metrics.

Classification metrics use a preliminary threshold of 0.50. Cost-based threshold selection comes later. These shuffled historical folds do not simulate future deployment: the untouched 2016 test sample is reserved for final evaluation. The population also consists of completed, granted loans, so these results do not establish performance for all applicants.

In [1]:
import numpy as np
import pandas as pd

from IPython.display import display

from preprocessing import load_split, clean, build_preprocessor

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    brier_score_loss,
)

RANDOM_STATE = 42
TARGET = "Default"

# Read the training sample using the shared function.
X_train, y_train = load_split("../data/train.csv")

# Keep the original columns for later error analysis.
model_data = X_train.copy().reset_index(drop=True)
model_data[TARGET] = y_train.reset_index(drop=True)

y = model_data[TARGET].astype(int)

print(f"Training rows: {len(y):,}; default rate: {y.mean():.2%}")

Training rows: 150,000; default rate: 18.46%


## 2. Preprocessing

The modeling notebook imports `load_split`, `clean`, and
`build_preprocessor` from the shared `preprocessing.py` module.

The full models use eight original inputs: FICO, income, DTI,
loan amount, employment length, home ownership, loan purpose,
and state.

The shared pipeline converts employment length to numeric years,
handles missing values with indicators, cleans DTI, creates
`loan_to_income`, applies log transformations and scaling,
one-hot encodes categorical features with rare-category grouping,
and applies smoothed target encoding to state.

The complete preprocessing pipeline is fitted separately within
each cross-validation training fold. State target encoding uses
additional internal cross-fitting. A feature matrix prepared
using the entire training sample is not used for cross-validation.

The FICO-only baseline uses only `fico_n`, with median imputation,
missing indicators, and standardization.

In [2]:
# Original input columns required by the shared preprocessing pipeline.
raw_feature_cols = [
    "fico_n",
    "revenue",
    "dti_n",
    "loan_amnt",
    "emp_length",
    "home_ownership_n",
    "purpose",
    "addr_state",
]

X = X_train[raw_feature_cols].copy().reset_index(drop=True)

# Full models use the shared preprocessing pipeline.
preprocessor = build_preprocessor()

# The baseline uses FICO only.
fico_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
            ),
            ("scaler", StandardScaler()),
        ]),
        ["fico_n"],
    ),
])

print("Raw model inputs:", raw_feature_cols)

Raw model inputs: ['fico_n', 'revenue', 'dti_n', 'loan_amnt', 'emp_length', 'home_ownership_n', 'purpose', 'addr_state']


## 3. Baseline and stratified five-fold comparison

The following configurations are fixed starting points, not tuned models. Cross-validation produces one out-of-fold probability per training loan. Each loan is predicted by a model that did not train on that loan. Random forests and Extra Trees parallelize tree fitting; model families and folds run sequentially to limit memory use.

Tree fitting uses at most four workers. Native numerical thread pools are limited to four during modeling to avoid oversubscription; this changes computational resource use, not the CV design.

In [3]:
from threadpoolctl import threadpool_limits
thread_limits = threadpool_limits(limits=4)

In [4]:
estimators = {
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=100, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=14, min_samples_leaf=20,
                                            max_features="sqrt", n_jobs=4, random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, max_depth=14, min_samples_leaf=20,
                                        max_features="sqrt", n_jobs=4, random_state=RANDOM_STATE),
    "Gradient boosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05,
        max_leaf_nodes=15, min_samples_leaf=50, l2_regularization=1.0, random_state=RANDOM_STATE),
}
pipelines = {
    "FICO-only baseline": Pipeline([
        ("preprocess", fico_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ])
}

for name, estimator in estimators.items():
    pipelines[name] = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", estimator),
    ])
    
def calculate_metrics(y_true, probability):
    prediction = (probability >= 0.50).astype(int)
    return {
        "ROC_AUC": roc_auc_score(y_true, probability),
        "AP": average_precision_score(y_true, probability),
        "Precision": precision_score(y_true, prediction, zero_division=0),
        "Recall": recall_score(y_true, prediction, zero_division=0),
        "F1": f1_score(y_true, prediction, zero_division=0),
        "Accuracy": accuracy_score(y_true, prediction),
        "Brier": brier_score_loss(y_true, probability),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X, y))
fold_records, oof_probabilities, first_fold_models = [], {}, {}
for name, pipeline in pipelines.items():
    print(f"Training: {name}", flush=True)
    oof = np.full(len(X), np.nan)
    for fold_number, (fit_idx, valid_idx) in enumerate(folds, start=1):
        fitted = clone(pipeline)
        fitted.fit(X.iloc[fit_idx], y.iloc[fit_idx])
        probability = fitted.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = probability
        print(f"  Fold {fold_number}/5: ROC-AUC={roc_auc_score(y.iloc[valid_idx], probability):.4f}", flush=True)
        fold_records.append({"Model": name, "Fold": fold_number,
                             **calculate_metrics(y.iloc[valid_idx], probability)})
        if fold_number == 1:
            first_fold_models[name] = fitted
            
    oof_probabilities[name] = oof

fold_results = pd.DataFrame(fold_records)
metric_names = ["ROC_AUC", "AP", "Precision", "Recall", "F1", "Accuracy", "Brier"]
cv_mean = fold_results.groupby("Model")[metric_names].mean()
cv_std = fold_results.groupby("Model")[metric_names].std()
ranking = cv_mean.sort_values(["ROC_AUC", "AP"], ascending=[False, False]).index
cv_report = pd.DataFrame(index=ranking)
for metric in metric_names:
    cv_report[metric] = [f"{cv_mean.loc[name, metric]:.4f} ± {cv_std.loc[name, metric]:.4f}"
                         for name in ranking]
display(cv_report)
baseline_auc = cv_mean.loc["FICO-only baseline", "ROC_AUC"]
baseline_ap = cv_mean.loc["FICO-only baseline", "AP"]
improvement = cv_mean[["ROC_AUC", "AP"]].copy()
improvement["ROC_AUC_gain"] = improvement["ROC_AUC"] - baseline_auc
improvement["AP_gain"] = improvement["AP"] - baseline_ap
display(improvement.loc[ranking].round(4))

Training: FICO-only baseline
  Fold 1/5: ROC-AUC=0.5840
  Fold 2/5: ROC-AUC=0.5909
  Fold 3/5: ROC-AUC=0.5887
  Fold 4/5: ROC-AUC=0.5952
  Fold 5/5: ROC-AUC=0.5931
Training: Logistic regression
  Fold 1/5: ROC-AUC=0.6611
  Fold 2/5: ROC-AUC=0.6559
  Fold 3/5: ROC-AUC=0.6582
  Fold 4/5: ROC-AUC=0.6613
  Fold 5/5: ROC-AUC=0.6683
Training: Decision tree
  Fold 1/5: ROC-AUC=0.6411
  Fold 2/5: ROC-AUC=0.6424
  Fold 3/5: ROC-AUC=0.6376
  Fold 4/5: ROC-AUC=0.6396
  Fold 5/5: ROC-AUC=0.6492
Training: Random forest
  Fold 1/5: ROC-AUC=0.6592
  Fold 2/5: ROC-AUC=0.6588
  Fold 3/5: ROC-AUC=0.6575
  Fold 4/5: ROC-AUC=0.6607
  Fold 5/5: ROC-AUC=0.6679
Training: Extra Trees
  Fold 1/5: ROC-AUC=0.6547
  Fold 2/5: ROC-AUC=0.6517
  Fold 3/5: ROC-AUC=0.6533
  Fold 4/5: ROC-AUC=0.6555
  Fold 5/5: ROC-AUC=0.6620
Training: Gradient boosting
  Fold 1/5: ROC-AUC=0.6625
  Fold 2/5: ROC-AUC=0.6586
  Fold 3/5: ROC-AUC=0.6596
  Fold 4/5: ROC-AUC=0.6623
  Fold 5/5: ROC-AUC=0.6706


,ROC_AUC,AP,Precision,Recall,F1,Accuracy,Brier
Model,,,,,,,
Gradient boosting,0.6627 ± 0.0047,0.3007 ± 0.0084,0.5394 ± 0.1014,0.0030 ± 0.0016,0.0059 ± 0.0032,0.8155 ± 0.0001,0.1428 ± 0.0006
Logistic regression,0.6610 ± 0.0047,0.2965 ± 0.0083,0.4745 ± 0.0726,0.0039 ± 0.0007,0.0078 ± 0.0015,0.8154 ± 0.0002,0.1430 ± 0.0006
Random forest,0.6608 ± 0.0041,0.2970 ± 0.0065,0.2833 ± 0.3891,0.0002 ± 0.0003,0.0004 ± 0.0005,0.8154 ± 0.0000,0.1431 ± 0.0004
Extra Trees,0.6554 ± 0.0039,0.2930 ± 0.0087,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.8154 ± 0.0000,0.1447 ± 0.0003
Decision tree,0.6420 ± 0.0044,0.2736 ± 0.0057,0.2207 ± 0.2055,0.0035 ± 0.0056,0.0069 ± 0.0108,0.8150 ± 0.0006,0.1448 ± 0.0005
FICO-only baseline,0.5904 ± 0.0043,0.2254 ± 0.0028,0.0000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,0.8154 ± 0.0000,0.1481 ± 0.0002


,ROC_AUC,AP,ROC_AUC_gain,AP_gain
Model,,,,
Gradient boosting,0.6627,0.3007,0.0724,0.0753
Logistic regression,0.6610,0.2965,0.0706,0.0712
Random forest,0.6608,0.2970,0.0704,0.0716
Extra Trees,0.6554,0.2930,0.0651,0.0676
Decision tree,0.6420,0.2736,0.0516,0.0483
FICO-only baseline,0.5904,0.2254,0.0000,0.0000


## 4. Held-out permutation importance

The top three non-baseline models by mean CV ROC-AUC form the provisional shortlist (AP breaks exact ties). Permutation importance measures the reduction in ROC-AUC after shuffling an original input feature through the complete fitted pipeline.

For computational efficiency, this diagnostic uses up to 3,000 rows from the first validation fold and five permutations per feature. The corresponding stored model did not train on these rows. The standard deviation describes permutation variability, not variability across folds. Correlated predictors can share importance; importance is predictive, not causal.

In [5]:
shortlist = [name for name in ranking if name != "FICO-only baseline"][:3]
print("Provisional shortlist:", shortlist)
_, importance_valid_idx = folds[0]
rng = np.random.default_rng(RANDOM_STATE)
importance_idx = rng.choice(importance_valid_idx, size=min(3000, len(importance_valid_idx)), replace=False)
importance_frames = []
for name in shortlist:
    result = permutation_importance(first_fold_models[name], X.iloc[importance_idx], y.iloc[importance_idx],
        scoring="roc_auc", n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
    table = pd.DataFrame({"Model": name, "Feature": X.columns,
                          "Importance_mean": result.importances_mean,
                          "Importance_std": result.importances_std}).sort_values("Importance_mean", ascending=False)
    importance_frames.append(table)
    print(f"\n{name}: top 10 features")
    display(table.head(10).round(4))
feature_importance_results = pd.concat(importance_frames, ignore_index=True)

Provisional shortlist: ['Gradient boosting', 'Logistic regression', 'Random forest']

Gradient boosting: top 10 features


,Model,Feature,Importance_mean,Importance_std
0,Gradient boosting,fico_n,0.0673,0.0063
3,Gradient boosting,loan_amnt,0.0657,0.0066
1,Gradient boosting,revenue,0.0340,0.0062
2,Gradient boosting,dti_n,0.0264,0.0054
6,Gradient boosting,purpose,0.0102,0.0043
7,Gradient boosting,addr_state,0.0083,0.0032
5,Gradient boosting,home_ownership_n,0.0055,0.0019
4,Gradient boosting,emp_length,0.0034,0.0014



Logistic regression: top 10 features


,Model,Feature,Importance_mean,Importance_std
3,Logistic regression,loan_amnt,0.0642,0.0049
0,Logistic regression,fico_n,0.0630,0.0061
1,Logistic regression,revenue,0.0340,0.0060
2,Logistic regression,dti_n,0.0255,0.0031
6,Logistic regression,purpose,0.0125,0.0049
7,Logistic regression,addr_state,0.0053,0.0029
5,Logistic regression,home_ownership_n,0.0028,0.0013
4,Logistic regression,emp_length,0.0027,0.0007



Random forest: top 10 features


,Model,Feature,Importance_mean,Importance_std
3,Random forest,loan_amnt,0.0684,0.0074
0,Random forest,fico_n,0.0554,0.0044
1,Random forest,revenue,0.0432,0.0064
2,Random forest,dti_n,0.0255,0.0039
5,Random forest,home_ownership_n,0.0087,0.0011
6,Random forest,purpose,0.0083,0.0026
7,Random forest,addr_state,0.0066,0.0026
4,Random forest,emp_length,0.0031,0.0013


## 5. Which loans does the model get wrong?

At the 0.50 threshold, a **false negative** is a defaulted loan predicted as non-default; a **false positive** is a fully paid loan predicted as default. FNR = FN / actual defaults; FPR = FP / actual non-defaults.

Compare the shortlisted models, then inspect the top-ranked candidate across FICO bands, predicted-risk bands, loan categories, loan amounts, and DTI where available. Quartile bands are descriptive only and are not fed back into model training. Always examine group counts alongside rates; small segments can give unstable rates.

The individual-case tables show confident mistakes for investigation. Their presence alone does not demonstrate bad labels or explain why a borrower defaulted.

In [6]:
# Compare OOF errors across shortlisted models.
error_records = []

for name in shortlist:
    probability = oof_probabilities[name]
    prediction = (probability >= 0.50).astype(int)
    actual = y.to_numpy()

    tn = int(((actual == 0) & (prediction == 0)).sum())
    fp = int(((actual == 0) & (prediction == 1)).sum())
    fn = int(((actual == 1) & (prediction == 0)).sum())
    tp = int(((actual == 1) & (prediction == 1)).sum())

    error_records.append({
        "Model": name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "FNR": fn / (fn + tp),
        "FPR": fp / (fp + tn),
    })

error_comparison = pd.DataFrame(error_records).set_index("Model")
display(error_comparison.round(4))


# Detailed analysis of the highest-ranked model.
best_model_name = shortlist[0]

# Deterministic cleaning for readable diagnostic fields only.
# Model predictions were generated using raw X and the fitted pipeline.
error_analysis = clean(X)

error_analysis["emp_length_missing"] = (
    error_analysis["emp_length"].isna().astype(int)
)

if "id" in model_data.columns:
    error_analysis.insert(0, "id", model_data["id"])

error_analysis["actual_default"] = y.to_numpy()

error_analysis["predicted_probability"] = (
    oof_probabilities[best_model_name]
)

error_analysis["predicted_default"] = (
    error_analysis["predicted_probability"] >= 0.50
).astype(int)

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual_default"] == 1)
        & (error_analysis["predicted_default"] == 0),

        (error_analysis["actual_default"] == 0)
        & (error_analysis["predicted_default"] == 1),
    ],
    ["False negative", "False positive"],
    default="Correct",
)


# FICO bands.
error_analysis["fico_band"] = pd.cut(
    error_analysis["fico_n"],
    bins=[-np.inf, 660, 700, 740, np.inf],
    labels=["<660", "660–699", "700–739", "740+"],
    right=False,
)

# Predicted-risk bands.
error_analysis["predicted_risk_band"] = pd.cut(
    error_analysis["predicted_probability"],
    bins=[0, 0.10, 0.20, 0.30, 0.50, 1],
    labels=["0–10%", "10–20%", "20–30%", "30–50%", "50–100%"],
    include_lowest=True,
)


def summarize_errors(data, group_col):
    working = data.copy()

    working["_fn"] = (
        working["error_type"] == "False negative"
    ).astype(int)

    working["_fp"] = (
        working["error_type"] == "False positive"
    ).astype(int)

    table = working.groupby(
        group_col,
        observed=True,
        dropna=False,
    ).agg(
        loans=("actual_default", "size"),
        defaults=("actual_default", "sum"),
        default_rate=("actual_default", "mean"),
        mean_predicted_risk=("predicted_probability", "mean"),
        false_negatives=("_fn", "sum"),
        false_positives=("_fp", "sum"),
    )

    table["FNR"] = (
        table["false_negatives"]
        / table["defaults"].replace(0, np.nan)
    )

    table["FPR"] = (
        table["false_positives"]
        / (table["loans"] - table["defaults"]).replace(0, np.nan)
    )

    return table.round(4)


# Categorical and missingness segments.
group_columns = [
    "fico_band",
    "predicted_risk_band",
    "home_ownership_n",
    "purpose",
    "emp_length",
    "emp_length_missing",
    "addr_state",
]

# Numeric segments.
for col in ["loan_amnt", "dti_n", "revenue", "loan_to_income"]:
    if error_analysis[col].nunique() > 1:
        band_col = f"{col}_quartile"

        error_analysis[band_col] = pd.qcut(
            error_analysis[col],
            q=4,
            duplicates="drop",
        )

        group_columns.append(band_col)


print(f"Detailed OOF error analysis: {best_model_name}")

error_group_tables = {}

for group_col in group_columns:
    print(f"\nErrors by {group_col}")

    error_group_tables[group_col] = summarize_errors(
        error_analysis,
        group_col,
    )

    display(error_group_tables[group_col])


# Most confident mistakes.
false_negatives = error_analysis.loc[
    error_analysis["error_type"] == "False negative"
].sort_values("predicted_probability")

false_positives = error_analysis.loc[
    error_analysis["error_type"] == "False positive"
].sort_values(
    "predicted_probability",
    ascending=False,
)

print("\nDefaulted loans assigned the lowest predicted risk:")
display(false_negatives.head(10))

print("\nFully paid loans assigned the highest predicted risk:")
display(false_positives.head(10))

,TN,FP,FN,TP,FNR,FPR
Model,,,,,,
Gradient boosting,122243,73,27602,82,0.9970,0.0006
Logistic regression,122194,122,27575,109,0.9961,0.0010
Random forest,122312,4,27679,5,0.9998,0.0000


Detailed OOF error analysis: Gradient boosting

Errors by fico_band


,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
fico_band,,,,,,,,
<660,90,28,0.3111,0.2261,28,0,1.0000,0.0000
660–699,93317,20223,0.2167,0.2158,20141,70,0.9959,0.0010
700–739,42010,6179,0.1471,0.1482,6179,3,1.0000,0.0001
740+,14583,1254,0.0860,0.0889,1254,0,1.0000,0.0000



Errors by predicted_risk_band


,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
predicted_risk_band,,,,,,,,
0–10%,23836,1632,0.0685,0.0735,1632,0,1.0,0.0
10–20%,70241,10592,0.1508,0.1492,10592,0,1.0,0.0
20–30%,39299,9557,0.2432,0.2430,9557,0,1.0,0.0
30–50%,16469,5821,0.3535,0.3536,5821,0,1.0,0.0
50–100%,155,82,0.5290,0.5247,0,73,0.0,1.0


...

,loans,defaults,default_rate,mean_predicted_risk,false_negatives,false_positives,FNR,FPR
loan_to_income_quartile,,,,,,,,
"(-0.00016700000000000005, 0.13]",37549,4807,0.1280,0.1286,4807,0,1.0000,0.0000
"(0.13, 0.203]",37451,5671,0.1514,0.1517,5671,0,1.0000,0.0000
"(0.203, 0.294]",37500,7263,0.1937,0.1939,7260,3,0.9996,0.0001
"(0.294, 0.83]",37500,9943,0.2651,0.2641,9864,70,0.9921,0.0025



Defaulted loans assigned the lowest predicted risk:


,id,fico_n,revenue,dti_n,loan_amnt,emp_length,home_ownership_n,purpose,addr_state,loan_to_income,...,actual_default,predicted_probability,predicted_default,error_type,fico_band,predicted_risk_band,loan_amnt_quartile,dti_n_quartile,revenue_quartile,loan_to_income_quartile
137549,64150409,812.0,92000.0,0.64,15000,10.0,MORTGAGE,home_improvement,OR,0.163043,...,1,0.021062,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]","(0.13, 0.203]"
33505,6817730,777.0,135000.0,2.46,10000,10.0,MORTGAGE,credit_card,TX,0.074074,...,1,0.023628,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]","(-0.00016700000000000005, 0.13]"
44276,10153588,777.0,170000.0,8.55,8400,7.0,MORTGAGE,credit_card,CA,0.049412,...,1,0.024582,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]","(-0.00016700000000000005, 0.13]"
66137,23864572,807.0,68000.0,9.18,14000,8.0,MORTGAGE,debt_consolidation,NJ,0.205882,...,1,0.025268,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(-0.001, 11.74]","(63500.0, 90000.0]","(0.203, 0.294]"
10921,1330328,792.0,90000.0,10.45,12000,10.0,MORTGAGE,credit_card,NJ,0.133333,...,1,0.026695,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(-0.001, 11.74]","(63500.0, 90000.0]","(0.13, 0.203]"
39963,9665076,792.0,135000.0,5.17,25000,6.0,MORTGAGE,credit_card,IL,0.185185,...,1,0.028927,0,False negative,740+,0–10%,"(20000.0, 35000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]","(0.13, 0.203]"
129339,61762341,752.0,90000.0,11.85,12500,10.0,MORTGAGE,credit_card,CO,0.138889,...,1,0.031808,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(11.74, 17.48]","(63500.0, 90000.0]","(0.13, 0.203]"
41471,7071343,797.0,90000.0,12.77,10400,9.0,MORTGAGE,debt_consolidation,CA,0.115556,...,1,0.031863,0,False negative,740+,0–10%,"(8000.0, 12000.0]","(11.74, 17.48]","(63500.0, 90000.0]","(-0.00016700000000000005, 0.13]"
134848,60902085,742.0,105000.0,11.29,24000,3.0,OWN,credit_card,OR,0.228571,...,1,0.032101,0,False negative,740+,0–10%,"(20000.0, 35000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]","(0.203, 0.294]"
82322,38614729,817.0,135000.0,8.59,18000,0.0,MORTGAGE,home_improvement,CA,0.133333,...,1,0.033252,0,False negative,740+,0–10%,"(12000.0, 20000.0]","(-0.001, 11.74]","(90000.0, 9500000.0]","(0.13, 0.203]"



Fully paid loans assigned the highest predicted risk:


,id,fico_n,revenue,dti_n,loan_amnt,emp_length,home_ownership_n,purpose,addr_state,loan_to_income,...,actual_default,predicted_probability,predicted_default,error_type,fico_band,predicted_risk_band,loan_amnt_quartile,dti_n_quartile,revenue_quartile,loan_to_income_quartile
90221,40530551,682.0,68000.0,36.80,23075,10.0,RENT,small_business,VA,0.339338,...,0,0.625684,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]","(0.294, 0.83]"
89135,40450887,662.0,35000.0,30.76,15000,4.0,RENT,small_business,PA,0.428571,...,0,0.575288,1,False positive,660–699,50–100%,"(12000.0, 20000.0]","(23.73, 83.4]","(1999.999, 45000.0]","(0.294, 0.83]"
139360,65571531,662.0,57077.0,32.91,20000,9.0,RENT,medical,NV,0.350404,...,0,0.565388,1,False positive,660–699,50–100%,"(12000.0, 20000.0]","(23.73, 83.4]","(45000.0, 63500.0]","(0.294, 0.83]"
111708,53484838,667.0,58000.0,34.68,24350,3.0,RENT,debt_consolidation,FL,0.419828,...,0,0.562890,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(45000.0, 63500.0]","(0.294, 0.83]"
117907,55201657,682.0,60000.0,28.88,25200,9.0,RENT,moving,AZ,0.420000,...,0,0.562557,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(45000.0, 63500.0]","(0.294, 0.83]"
137475,64360949,682.0,91000.0,32.40,35000,10.0,RENT,other,NV,0.384615,...,0,0.551426,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(90000.0, 9500000.0]","(0.294, 0.83]"
103597,48564893,662.0,52000.0,33.42,23400,7.0,RENT,debt_consolidation,NV,0.450000,...,0,0.550199,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(45000.0, 63500.0]","(0.294, 0.83]"
7640,1039015,707.0,60000.0,21.34,27050,4.0,RENT,small_business,FL,0.450833,...,0,0.548993,1,False positive,700–739,50–100%,"(20000.0, 35000.0]","(17.48, 23.73]","(45000.0, 63500.0]","(0.294, 0.83]"
110295,50920161,662.0,70000.0,38.21,24000,10.0,RENT,debt_consolidation,CA,0.342857,...,0,0.544142,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]","(0.294, 0.83]"
71089,32659627,662.0,68000.0,38.16,33000,10.0,RENT,debt_consolidation,TX,0.485294,...,0,0.543502,1,False positive,660–699,50–100%,"(20000.0, 35000.0]","(23.73, 83.4]","(63500.0, 90000.0]","(0.294, 0.83]"


## 6. Shortlist for the next modeling stage

The table below assembles the top three candidates' CV performance, gains over the FICO baseline, and out-of-fold error rates. These are provisional selections for further tuning. Review whether small performance gains justify complexity, and consider probability accuracy, stability, and operational error costs. A candidate can underperform the baseline; the gain columns make this visible.

Selection and diagnostic analysis reuse the historical CV results, so the winning CV score is not an unbiased final estimate. Final modeling choices and threshold selection must use training data only, followed by one final evaluation on the untouched 2016 test sample.

In [7]:
shortlist_summary = cv_report.loc[shortlist].copy()
shortlist_summary["ROC_AUC_gain_vs_baseline"] = (cv_mean.loc[shortlist, "ROC_AUC"] - baseline_auc).round(4)
shortlist_summary["AP_gain_vs_baseline"] = (cv_mean.loc[shortlist, "AP"] - baseline_ap).round(4)
shortlist_summary["OOF_FNR_at_0.50"] = error_comparison.loc[shortlist, "FNR"].round(4)
shortlist_summary["OOF_FPR_at_0.50"] = error_comparison.loc[shortlist, "FPR"].round(4)
display(shortlist_summary)
print("Models shortlisted for further tuning:")
for position, name in enumerate(shortlist, start=1):
    print(f"{position}. {name}")

,ROC_AUC,AP,Precision,Recall,F1,Accuracy,Brier,ROC_AUC_gain_vs_baseline,AP_gain_vs_baseline,OOF_FNR_at_0.50,OOF_FPR_at_0.50
Model,,,,,,,,,,,
Gradient boosting,0.6627 ± 0.0047,0.3007 ± 0.0084,0.5394 ± 0.1014,0.0030 ± 0.0016,0.0059 ± 0.0032,0.8155 ± 0.0001,0.1428 ± 0.0006,0.0724,0.0753,0.9970,0.0006
Logistic regression,0.6610 ± 0.0047,0.2965 ± 0.0083,0.4745 ± 0.0726,0.0039 ± 0.0007,0.0078 ± 0.0015,0.8154 ± 0.0002,0.1430 ± 0.0006,0.0706,0.0712,0.9961,0.0010
Random forest,0.6608 ± 0.0041,0.2970 ± 0.0065,0.2833 ± 0.3891,0.0002 ± 0.0003,0.0004 ± 0.0005,0.8154 ± 0.0000,0.1431 ± 0.0004,0.0704,0.0716,0.9998,0.0000


Models shortlisted for further tuning:
1. Gradient boosting
2. Logistic regression
3. Random forest


## 7. Results discussion — shared preprocessing v2

All six configurations completed identical stratified five-fold CV on the 150,000-loan training sample using the shared preprocessing module. The preprocessor was fitted inside each outer training fold, including internally cross-fitted state target encoding. No globally prepared feature matrix or test sample was used for CV.

| Model | ROC-AUC (mean ± sample std) | Mean AP | Mean Brier |
|---|---:|---:|---:|
| Gradient boosting | 0.6627 ± 0.0047 | 0.3007 | 0.1428 |
| Logistic regression | 0.6610 ± 0.0047 | 0.2965 | 0.1430 |
| Random forest | 0.6608 ± 0.0041 | 0.2970 | 0.1431 |
| Extra Trees | 0.6554 ± 0.0039 | 0.2930 | 0.1447 |
| Decision tree | 0.6420 ± 0.0044 | 0.2736 | 0.1448 |
| FICO-only baseline | 0.5904 ± 0.0043 | 0.2254 | 0.1481 |

**Shortlist:** Gradient boosting, Logistic regression, Random forest. Selection uses mean ROC-AUC, with AP as an exact-tie breaker; other metrics and errors support review rather than automatic selection. Fold-to-fold standard deviations are not confidence intervals or a significance test.

For **Gradient boosting**, the four largest mean held-out permutation AUC reductions were `fico_n` (0.0673), `loan_amnt` (0.0657), `revenue` (0.0340), `dti_n` (0.0264). Importance is measured on original inputs; shuffling loan amount or income also changes the derived loan-to-income ratio. These values do not establish causation.

At the initial **0.50 threshold**, Gradient boosting detected **82 of 27,684 defaults** and missed **27,602** (FNR **99.70%**), with **73** false positives. Review recall and conditional error rates before treating this as an approval rule; a high accuracy can conceal missed defaults in an imbalanced sample. The segment tables above show the actual group counts and error rates.

These results replace those from the earlier independent preprocessing recipe; the feature set and transformations have changed. The shared module is used unchanged. Hyperparameter tuning and cost-based threshold selection remain future steps using training data only, followed by final evaluation on the untouched 2016 sample. No final estimator has yet been refitted on all training rows.
